In [1]:
"""
PASSO 8 — Portfólio Long-Short baseado no ISRA (Setor + Individual).

Inclui duas versões do ISRA:
  - ISRA_Setor: melhor modelo por SETOR (replicação do artigo)
  - ISRA_Ind:   melhor modelo por AÇÃO INDIVIDUAL (contribuição original)

Para ações In-Sample: TODIM individual roda no InS-TraD.
Para ações Out-of-Sample: TODIM individual roda no OutS-TraD.
(Análogo ao que os autores fazem no nível setorial.)

Uso:
    python 08_portfolio.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURAÇÃO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"
INDEX_FILE = BASE_DIR / ".IBOV_Index.csv"

ALGORITHMS = ["LR", "SVM", "NB", "CART", "RF", "XGB",
              "MLP", "DBN", "SAE", "RNN", "LSTM", "GRU"]

METRICS = ["WR", "ARR", "ASR", "MDD", "CAR", "SOR", "MSR", "ART"]
COST_INDICATORS = ["MDD", "ART"]
Q_WEIGHTS = [1, 3, 4, 5, 6, 7, 2, 2]
ALPHA_T, BETA_T, LAMBDA_T = 0.88, 0.88, 2.25

ANNUALIZATION = 250

EVAL_SCENARIOS = {
    "InS-TesD":  {"Sample": "In",  "Start": "2020-01-02", "End": "2020-06-30"},
    "OutS-TesD": {"Sample": "Out", "Start": "2020-01-02", "End": "2020-06-30"},
}
CALIB = {"Sample": "In", "Start": "2018-01-02", "End": "2019-12-31"}
# ========================================================


# ============================================================
#              MÉTRICAS DE PERFORMANCE
# ============================================================

def win_rate(r):
    nz = r[r != 0]
    return len(nz[nz > 0]) / len(nz) if len(nz) > 0 else 0.0

def ann_return(r, scale=ANNUALIZATION):
    return r.mean() * scale

def ann_sharpe(r, scale=ANNUALIZATION):
    s = r.std(ddof=1)
    return np.sqrt(scale) * r.mean() / s if s > 0 else 0.0

def max_drawdown(r):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return abs(dd.min())

def calmar(r):
    mdd = max_drawdown(r)
    return ann_return(r) / mdd if mdd > 0 else 0.0

def sortino(r, mar=0.0):
    down = r[r < mar]
    ds = np.sqrt((down ** 2).mean()) if len(down) > 0 else 0.0
    return (r.mean() - mar) / ds if ds > 0 else 0.0

def adj_sharpe(r):
    asr = ann_sharpe(r)
    s = r.std(ddof=1)
    if s == 0 or len(r) < 4: return asr
    m = r.mean()
    skew = ((r - m) ** 3).mean() / (s ** 3)
    kurt = ((r - m) ** 4).mean() / (s ** 4) - 3
    return asr * (1 + (skew / 6) * asr - (kurt / 24) * asr ** 2)

def avg_recovery(r):
    cum = (1 + r).cumprod()
    peak = cum.cummax()
    in_dd = cum < peak
    periods, length = [], 0
    for v in in_dd:
        if v: length += 1
        else:
            if length > 0: periods.append(length)
            length = 0
    if length > 0: periods.append(length)
    return np.mean(periods) if periods else 0.0

def compute_all_metrics(r):
    r = r.dropna()
    if len(r) < 10: return [np.nan] * 8
    return [win_rate(r), ann_return(r), ann_sharpe(r), max_drawdown(r),
            calmar(r), sortino(r), adj_sharpe(r), avg_recovery(r)]

METRIC_NAMES = ["WR", "ARR", "ASR", "MDD", "CAR", "SOR", "MSR", "ART"]


# ============================================================
#              TODIM
# ============================================================

def maxmin_norm(x):
    mn, mx = np.nanmin(x), np.nanmax(x)
    return np.zeros_like(x) if mx == mn else (x - mn) / (mx - mn)

def cost_norm(x):
    mn, mx = np.nanmin(x), np.nanmax(x)
    return np.zeros_like(x) if mx == mn else (mx - x) / (mx - mn)

def prospect_value(diff, w):
    if diff > 0:
        return (w * diff) ** ALPHA_T
    elif diff == 0:
        return 0.0
    else:
        return -LAMBDA_T * ((-diff) / w) ** BETA_T

def todim_rank(decision_matrix_df):
    """Aplica TODIM a uma matriz de decisão. Retorna lista ordenada."""
    mat = decision_matrix_df.copy()
    mat = mat.replace([np.inf, -np.inf], np.nan)

    global_mean = np.nanmean(mat.values)
    global_std = np.nanstd(mat.values)
    if global_std == 0: global_std = 1.0
    for col in mat.columns:
        mask = mat[col].isna()
        if mask.any():
            mat.loc[mask, col] = np.random.normal(global_mean, global_std, mask.sum())

    for col in METRICS:
        if col in mat.columns:
            if col in COST_INDICATORS:
                mat[col] = cost_norm(mat[col].values)
            else:
                mat[col] = maxmin_norm(mat[col].values)

    q = np.array(Q_WEIGHTS, dtype=float)
    w = q / q.max()
    N = len(mat)
    M = len(METRICS)
    algo_names = mat.index.tolist()

    phi_total = np.zeros(N)
    for j in range(N):
        for m_idx in range(M):
            col = METRICS[m_idx]
            if col not in mat.columns: continue
            for k in range(N):
                diff = mat.iloc[j][col] - mat.iloc[k][col]
                phi_total[j] += prospect_value(diff, w[m_idx])

    if phi_total.max() == phi_total.min():
        return algo_names
    xi = (phi_total - phi_total.min()) / (phi_total.max() - phi_total.min())
    ranking = pd.Series(xi, index=algo_names).sort_values(ascending=False)
    return ranking.index.tolist()


def get_best_model_per_stock(base_dir, sec_names_path, sample, scenario_label):
    """
    Roda TODIM para cada ação individual.
    Retorna dict {code: best_algo}.
    """
    codes_df = pd.read_csv(sec_names_path, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    industries = sorted(codes_df["industry"].str.strip().unique())

    best_per_stock = {}

    for ind in industries:
        mask = (codes_df["industry"].str.strip() == ind) & \
               (codes_df["Sample"].str.strip() == sample)
        codes = codes_df.loc[mask, "Code"].str.strip().str.upper().tolist()

        for code in codes:
            stock_metrics = {}
            for algo in ALGORITHMS:
                perf_file = base_dir / f".Performance_{algo}_{ind}_{sample}_{scenario_label}.csv"
                if not perf_file.exists():
                    continue
                try:
                    perf = pd.read_csv(perf_file, index_col=0, encoding="utf-8-sig")
                    if code not in perf.index:
                        continue
                    row = perf.loc[code]
                    metrics_vals = []
                    for metric in METRICS:
                        col_trade = f"{metric}Trade"
                        if col_trade in row.index:
                            metrics_vals.append(row[col_trade])
                        else:
                            metrics_vals.append(np.nan)
                    stock_metrics[algo] = metrics_vals
                except Exception:
                    continue

            if len(stock_metrics) < 2:
                if stock_metrics:
                    best_per_stock[code] = list(stock_metrics.keys())[0]
                continue

            dm = pd.DataFrame(stock_metrics, index=METRICS).T
            try:
                ranking = todim_rank(dm)
                best_per_stock[code] = ranking[0]
            except Exception:
                best_per_stock[code] = list(stock_metrics.keys())[0]

    return best_per_stock


# ============================================================
#              FUNÇÕES AUXILIARES
# ============================================================

def load_index_returns(index_path):
    df = pd.read_csv(index_path, encoding="utf-8-sig")
    df[df.columns[0]] = pd.to_datetime(df[df.columns[0]], errors="coerce")
    df = df.dropna(subset=[df.columns[0]]).sort_values(df.columns[0]).set_index(df.columns[0])
    return df["Close"].astype(float).pct_change().dropna()


def load_todim_top1(base_dir):
    rank_file = base_dir / "Ranking_TODIM_In_InS.csv"
    if not rank_file.exists():
        print(f"[ERRO] {rank_file.name} não encontrado.")
        return None
    rank_df = pd.read_csv(rank_file, index_col=0, encoding="utf-8-sig")
    top1 = {}
    for col in rank_df.columns:
        val = rank_df[col].dropna()
        if len(val) > 0:
            top1[col] = val.iloc[0]
    return top1


def compute_strategy_returns(base_dir, sec_names_path, index_ret,
                             model_map, sample, start, end, by="sector"):
    """
    Calcula retornos diários.
    model_map: {sector: algo} se by="sector", ou {code: algo} se by="stock"
    """
    codes_df = pd.read_csv(sec_names_path, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]

    start_dt, end_dt = pd.to_datetime(start), pd.to_datetime(end)

    all_trade, all_bah, all_index = [], [], []
    industries = sorted(codes_df["industry"].str.strip().unique())

    for sector in industries:
        mask = (codes_df["industry"].str.strip() == sector) & \
               (codes_df["Sample"].str.strip() == sample)
        codes = codes_df.loc[mask, "Code"].str.strip().str.upper().tolist()
        if not codes: continue

        sector_trade, sector_bah, sector_idx = [], [], []

        for code in codes:
            if by == "sector":
                algo = model_map.get(sector)
            else:
                algo = model_map.get(code)

            if algo is None: continue

            sig_file = base_dir / f"{code}_TradeSignal_{algo}.csv"
            raw_file = base_dir / f"{code}_New.csv"
            if not sig_file.exists() or not raw_file.exists(): continue

            try:
                sig_df = pd.read_csv(sig_file, encoding="utf-8-sig")
                sig_df[sig_df.columns[0]] = pd.to_datetime(sig_df[sig_df.columns[0]], errors="coerce")
                sig_df = sig_df.dropna(subset=[sig_df.columns[0]])
                signal = pd.Series(sig_df.iloc[:, 1].values,
                                   index=pd.to_datetime(sig_df.iloc[:, 0].values))
                if len(signal) < 2: continue
                signal = signal.shift(1)
                signal.iloc[0] = 0

                raw_df = pd.read_csv(raw_file, encoding="utf-8-sig")
                raw_df[raw_df.columns[0]] = pd.to_datetime(raw_df[raw_df.columns[0]], errors="coerce")
                raw_df = raw_df.dropna(subset=[raw_df.columns[0]]).sort_values(raw_df.columns[0])
                raw_df = raw_df.set_index(raw_df.columns[0])
                raw_ret = raw_df["Close"].astype(float).pct_change()

                trade_ret = signal * raw_ret
                sector_trade.append(trade_ret)
                sector_bah.append(raw_ret)
                sector_idx.append(index_ret)
            except Exception:
                continue

        if sector_trade:
            trade_df = pd.concat(sector_trade, axis=1).loc[start_dt:end_dt]
            bah_df = pd.concat(sector_bah, axis=1).loc[start_dt:end_dt]
            idx_df = pd.concat(sector_idx, axis=1).loc[start_dt:end_dt]
            all_trade.append(trade_df.mean(axis=1))
            all_bah.append(bah_df.mean(axis=1))
            all_index.append(idx_df.mean(axis=1))

    if not all_trade:
        return None, None, None

    isra_ret = pd.concat(all_trade, axis=1).mean(axis=1).dropna()
    bah_ret = pd.concat(all_bah, axis=1).mean(axis=1).dropna()
    idx_ret = pd.concat(all_index, axis=1).mean(axis=1).dropna()

    common = isra_ret.index.intersection(bah_ret.index).intersection(idx_ret.index)
    return isra_ret.loc[common], bah_ret.loc[common], idx_ret.loc[common]


def kelly_ratio(returns):
    m = returns.mean()
    v = returns.var(ddof=1)
    kr = np.where(v > 0, m / v, 0)
    kr = np.maximum(kr, 0)
    s = kr.sum()
    return kr / s if s > 0 else np.ones(len(kr)) / len(kr)

def risk_parity_weights(returns):
    vols = returns.std(ddof=1)
    inv_vol = np.where(vols > 0, 1.0 / vols, 0)
    s = inv_vol.sum()
    return inv_vol / s if s > 0 else np.ones(len(inv_vol)) / len(inv_vol)


def build_all_spreads(isra_setor, isra_ind, bah_ret, idx_ret):
    """Constrói DataFrame com todos os 12 spreads possíveis."""
    return pd.DataFrame({
        "BAH-ISRA_Set":      bah_ret.values - isra_setor.values,
        "BAH-ISRA_Ind":      bah_ret.values - isra_ind.values,
        "BAH-Index":         bah_ret.values - idx_ret.values,
        "Index-ISRA_Set":    idx_ret.values - isra_setor.values,
        "Index-ISRA_Ind":    idx_ret.values - isra_ind.values,
        "Index-BAH":         idx_ret.values - bah_ret.values,
        "ISRA_Set-BAH":      isra_setor.values - bah_ret.values,
        "ISRA_Set-Index":    isra_setor.values - idx_ret.values,
        "ISRA_Ind-BAH":      isra_ind.values - bah_ret.values,
        "ISRA_Ind-Index":    isra_ind.values - idx_ret.values,
        "ISRA_Ind-ISRA_Set": isra_ind.values - isra_setor.values,
        "ISRA_Set-ISRA_Ind": isra_setor.values - isra_ind.values,
    }, index=isra_setor.index)


def calibrate_portfolio(isra_setor, isra_ind, bah_ret, idx_ret):
    """Monta spreads e calibra portfólios."""
    spreads = build_all_spreads(isra_setor, isra_ind, bah_ret, idx_ret)

    sharpes = {}
    for col in spreads.columns:
        s = spreads[col].std(ddof=1)
        sharpes[col] = np.sqrt(ANNUALIZATION) * spreads[col].mean() / s if s > 0 else 0.0

    print(f"\n  Sharpes dos 12 spreads:")
    for name, val in sorted(sharpes.items(), key=lambda x: -x[1]):
        marker = " <<<" if val > 0 else ""
        print(f"    {name:>20}: {val:.4f}{marker}")

    selected = [name for name, val in sharpes.items() if val > 0]
    if not selected:
        print("  [AVISO] Nenhum spread com Sharpe > 0.")
        return None, None, None

    print(f"\n  Spreads selecionados ({len(selected)}): {selected}")
    ls_returns = spreads[selected]

    n = len(selected)
    w_ewp = np.ones(n) / n
    sharpe_sel = np.array([sharpes[s] for s in selected])
    w_asrwp = sharpe_sel / sharpe_sel.sum()
    w_kwp = kelly_ratio(ls_returns)
    w_rpwp = risk_parity_weights(ls_returns)

    weights = pd.DataFrame({
        "Spread": selected,
        "EWP": w_ewp,
        "ASRWP": w_asrwp,
        "KWP": w_kwp,
        "RPWP": w_rpwp,
    })

    return selected, weights, ls_returns


def compute_portfolio_returns(ls_returns, weights_df, selected):
    ls_mat = ls_returns[selected].values
    portfolios = {}
    for port_name in ["EWP", "ASRWP", "KWP", "RPWP"]:
        w = weights_df[port_name].values
        port_ret = pd.Series(ls_mat @ w, index=ls_returns.index)
        portfolios[port_name] = port_ret
    return portfolios


def evaluate_and_save(strategies, scenario_name, base_dir):
    rows = []
    for name, ret in strategies.items():
        metrics = compute_all_metrics(ret)
        rows.append({"Strategy": name, **dict(zip(METRIC_NAMES, metrics))})

    perf_df = pd.DataFrame(rows).set_index("Strategy")
    perf_file = base_dir / f"Portfolio_Performance_{scenario_name}.csv"
    perf_df.to_csv(perf_file, encoding="utf-8-sig")

    print(f"\n  Performance ({scenario_name}):")
    print(perf_df.round(4).to_string())

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(14, 7))
        for name, ret in strategies.items():
            cumret = (1 + ret).cumprod()
            lw = 2.0 if name in ["ISRA_Setor", "ISRA_Ind"] else 1.2
            ls = "--" if name == "ISRA_Ind" else "-"
            ax.plot(cumret.index, cumret.values, label=name, linewidth=lw, linestyle=ls)

        ax.set_xlabel("Data")
        ax.set_ylabel("Retorno Cumulativo")
        ax.set_title(f"Retorno Cumulativo das Estratégias — {scenario_name}")
        ax.legend(loc="best")
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        plot_file = base_dir / f"Portfolio_CumulativeReturn_{scenario_name}.png"
        fig.savefig(plot_file, dpi=150)
        plt.close(fig)
        print(f"  Gráfico salvo: {plot_file.name}")
    except ImportError:
        pass

    return perf_df


# ============================================================
#                    EXECUÇÃO PRINCIPAL
# ============================================================

def main():
    print("=" * 60)
    print("PASSO 8: Portfólio (ISRA Setor + ISRA Individual)")
    print("=" * 60)

    # 1. Modelo ótimo por setor (TODIM InS-TraD)
    top1_setor = load_todim_top1(BASE_DIR)
    if top1_setor is None: return

    print(f"\nISRA Setor (top-1 por setor, InS-TraD):")
    for sector, algo in top1_setor.items():
        print(f"  {sector}: {algo}")

    # 2. Modelo ótimo por ação individual — In-Sample (InS-TraD)
    print(f"\nCalculando TODIM por ação — In-Sample (In_InS)...")
    best_per_stock_in = get_best_model_per_stock(
        BASE_DIR, SEC_NAMES, "In", "In_InS")
    print(f"  Total: {len(best_per_stock_in)} ações In-Sample")
    freq_in = Counter(best_per_stock_in.values())
    for algo, count in freq_in.most_common():
        print(f"    {algo}: {count} ações ({count/len(best_per_stock_in)*100:.0f}%)")

    # 3. Modelo ótimo por ação individual — Out-of-Sample (OutS-TraD)
    print(f"\nCalculando TODIM por ação — Out-of-Sample (Out_InS)...")
    best_per_stock_out = get_best_model_per_stock(
        BASE_DIR, SEC_NAMES, "Out", "Out_InS")
    print(f"  Total: {len(best_per_stock_out)} ações Out-of-Sample")
    freq_out = Counter(best_per_stock_out.values())
    for algo, count in freq_out.most_common():
        print(f"    {algo}: {count} ações ({count/len(best_per_stock_out)*100:.0f}%)")

    # Salvar ambos
    bp_rows = []
    for code, algo in best_per_stock_in.items():
        bp_rows.append({"Code": code, "Sample": "In", "BestModel": algo})
    for code, algo in best_per_stock_out.items():
        bp_rows.append({"Code": code, "Sample": "Out", "BestModel": algo})
    pd.DataFrame(bp_rows).to_csv(
        BASE_DIR / "Best_Model_Per_Stock.csv", index=False, encoding="utf-8-sig")

    # 4. Carregar índice
    index_ret = load_index_returns(INDEX_FILE)

    # ========================================================
    # FASE 1: CALIBRAÇÃO (InS-TraD)
    # ========================================================
    print(f"\n{'='*60}")
    print("FASE 1: Calibração no InS-TraD")
    print("=" * 60)

    isra_setor, bah_calib, idx_calib = compute_strategy_returns(
        BASE_DIR, SEC_NAMES, index_ret, top1_setor,
        CALIB["Sample"], CALIB["Start"], CALIB["End"], by="sector")

    isra_ind, _, _ = compute_strategy_returns(
        BASE_DIR, SEC_NAMES, index_ret, best_per_stock_in,
        CALIB["Sample"], CALIB["Start"], CALIB["End"], by="stock")

    if isra_setor is None or isra_ind is None:
        print("[ERRO] Sem dados para calibração.")
        return

    common = isra_setor.index.intersection(isra_ind.index) \
                .intersection(bah_calib.index).intersection(idx_calib.index)
    isra_setor = isra_setor.loc[common]
    isra_ind = isra_ind.loc[common]
    bah_calib = bah_calib.loc[common]
    idx_calib = idx_calib.loc[common]

    print(f"  Dias de negociação: {len(common)}")

    selected, weights_df, ls_calib = calibrate_portfolio(
        isra_setor, isra_ind, bah_calib, idx_calib)

    if selected is None:
        print("[ERRO] Impossível construir portfólio.")
        return

    weights_df.to_csv(BASE_DIR / "Portfolio_Weights.csv",
                      index=False, encoding="utf-8-sig")

    portfolios_calib = compute_portfolio_returns(ls_calib, weights_df, selected)
    strategies_calib = {
        "Index": idx_calib,
        "BAH": bah_calib,
        "ISRA_Setor": isra_setor,
        "ISRA_Ind": isra_ind,
    }
    strategies_calib.update(portfolios_calib)
    evaluate_and_save(strategies_calib, "InS-TraD", BASE_DIR)

    # ========================================================
    # FASE 2: AVALIAÇÃO (InS-TesD e OutS-TesD)
    # ========================================================
    for scen_name, scen_params in EVAL_SCENARIOS.items():
        print(f"\n{'='*60}")
        print(f"FASE 2: Avaliação no {scen_name}")
        print("=" * 60)

        # Escolher o dict de melhor modelo por ação correto
        if scen_params["Sample"] == "In":
            best_stock_map = best_per_stock_in
        else:
            best_stock_map = best_per_stock_out

        isra_setor_eval, bah_eval, idx_eval = compute_strategy_returns(
            BASE_DIR, SEC_NAMES, index_ret, top1_setor,
            scen_params["Sample"], scen_params["Start"], scen_params["End"],
            by="sector")

        isra_ind_eval, _, _ = compute_strategy_returns(
            BASE_DIR, SEC_NAMES, index_ret, best_stock_map,
            scen_params["Sample"], scen_params["Start"], scen_params["End"],
            by="stock")

        if isra_setor_eval is None or isra_ind_eval is None:
            print(f"  [AVISO] Sem dados para {scen_name}.")
            continue

        common = isra_setor_eval.index.intersection(isra_ind_eval.index) \
                    .intersection(bah_eval.index).intersection(idx_eval.index)
        isra_setor_eval = isra_setor_eval.loc[common]
        isra_ind_eval = isra_ind_eval.loc[common]
        bah_eval = bah_eval.loc[common]
        idx_eval = idx_eval.loc[common]

        print(f"  Dias de negociação: {len(common)}")

        all_spreads = build_all_spreads(
            isra_setor_eval, isra_ind_eval, bah_eval, idx_eval)

        # Mesmos spreads e pesos do InS-TraD
        available = [s for s in selected if s in all_spreads.columns]
        ls_eval = all_spreads[available]
        # Ajustar pesos se algum spread não está disponível
        w_df_eval = weights_df[weights_df["Spread"].isin(available)].copy()
        if len(w_df_eval) < len(weights_df):
            for col in ["EWP", "ASRWP", "KWP", "RPWP"]:
                s = w_df_eval[col].sum()
                if s > 0:
                    w_df_eval[col] = w_df_eval[col] / s

        portfolios_eval = compute_portfolio_returns(ls_eval, w_df_eval, available)

        strategies_eval = {
            "Index": idx_eval,
            "BAH": bah_eval,
            "ISRA_Setor": isra_setor_eval,
            "ISRA_Ind": isra_ind_eval,
        }
        strategies_eval.update(portfolios_eval)
        evaluate_and_save(strategies_eval, scen_name, BASE_DIR)

    print(f"\n{'='*60}")
    print("Concluído!")
    print(f"Arquivos gerados:")
    print(f"  - Best_Model_Per_Stock.csv")
    print(f"  - Portfolio_Weights.csv")
    print(f"  - Portfolio_Performance_*.csv (3 cenários)")
    print(f"  - Portfolio_CumulativeReturn_*.png (3 gráficos)")


if __name__ == "__main__":
    main()

PASSO 8: Portfólio (ISRA Setor + ISRA Individual)

ISRA Setor (top-1 por setor, InS-TraD):
  BM: RF
  CC: SVM
  CNC: NB
  COM: RNN
  FIN: RF
  GCS: CART
  OGB: NB
  UTI: GRU

Calculando TODIM por ação — In-Sample (In_InS)...
  Total: 190 ações In-Sample
    RF: 27 ações (14%)
    SVM: 26 ações (14%)
    XGB: 24 ações (13%)
    NB: 23 ações (12%)
    LR: 20 ações (11%)
    CART: 18 ações (9%)
    LSTM: 13 ações (7%)
    RNN: 11 ações (6%)
    MLP: 10 ações (5%)
    SAE: 7 ações (4%)
    GRU: 6 ações (3%)
    DBN: 5 ações (3%)

Calculando TODIM por ação — Out-of-Sample (Out_InS)...
  Total: 49 ações Out-of-Sample
    SVM: 7 ações (14%)
    RF: 6 ações (12%)
    NB: 6 ações (12%)
    LR: 5 ações (10%)
    CART: 5 ações (10%)
    MLP: 5 ações (10%)
    XGB: 4 ações (8%)
    LSTM: 4 ações (8%)
    GRU: 2 ações (4%)
    RNN: 2 ações (4%)
    SAE: 2 ações (4%)
    DBN: 1 ações (2%)

FASE 1: Calibração no InS-TraD
  Dias de negociação: 493

  Sharpes dos 12 spreads:
       ISRA_Ind-ISRA_Set: 1